# Mining the Future: Black Friday Sales Insights
**Data Mining – Year 1 Summative Assessment**

This notebook covers all 8 stages:
1. Project Scope Definition
2. Data Cleaning & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Clustering Analysis
5. Association Rule Mining
6. Anomaly Detection
7. Insights & Reporting
8. (Deployment done via app.py + Streamlit Cloud)

## Stage 1: Define the Project Scope

**Dataset:** Black Friday retail transactions from InsightMart Analytics.

**Columns:** User_ID, Product_ID, Gender, Age, Occupation, City_Category, Stay_In_Current_City_Years, Marital_Status, Product_Category_1, Product_Category_2, Product_Category_3, Purchase

**Objectives:**
- Understand shopping preferences: which age/gender groups spend the most
- Segment customers into clusters (e.g. Budget Shoppers, Premium Buyers)
- Identify cross-selling opportunities via association rule mining
- Detect anomalies (unusually high spenders)
- Deploy insights via a Streamlit dashboard

## Stage 2: Data Cleaning & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Load dataset ──────────────────────────────────────────────────────────────
# Download from: https://drive.google.com/drive/folders/13DxtCVj3S_AAYXG5THw2mmr6_VA1N3L9
df = pd.read_csv('train.csv')   # rename your downloaded file to train.csv
print('Shape:', df.shape)
df.head()

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
print('Missing values before:')
print(df.isnull().sum())

# Fill missing product categories with 0 (means not purchased)
df['Product_Category_2'] = df['Product_Category_2'].fillna(0).astype(int)
df['Product_Category_3'] = df['Product_Category_3'].fillna(0).astype(int)

print('\nMissing values after:')
print(df.isnull().sum())

In [ ]:
# ── Duplicates ────────────────────────────────────────────────────────────────
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates()

# ── Encode Gender ─────────────────────────────────────────────────────────────
df['Gender_Encoded'] = df['Gender'].map({'M': 0, 'F': 1})

# ── Encode Age ────────────────────────────────────────────────────────────────
age_map = {'0-17': 1, '18-25': 2, '26-35': 3, '36-45': 4, '46-50': 5, '51-55': 6, '55+': 7}
df['Age_Encoded'] = df['Age'].map(age_map)

# ── Encode City Category ──────────────────────────────────────────────────────
df['City_Encoded'] = df['City_Category'].map({'A': 1, 'B': 2, 'C': 3})

# ── Encode Stay years (treat '4+' as 4) ──────────────────────────────────────
df['Stay_Encoded'] = df['Stay_In_Current_City_Years'].replace('4+', 4).astype(int)

# ── Normalize Purchase ───────────────────────────────────────────────────────
scaler = StandardScaler()
df['Purchase_Normalized'] = scaler.fit_transform(df[['Purchase']])

print('\nCleaned dataset shape:', df.shape)
df.head()

## Stage 3: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Purchase distribution by Age
age_order = ['0-17','18-25','26-35','36-45','46-50','51-55','55+']
df.boxplot(column='Purchase', by='Age', ax=axes[0], order=age_order)
axes[0].set_title('Purchase Amount by Age Group')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Purchase (USD)')

# Purchase distribution by Gender
df.boxplot(column='Purchase', by='Gender', ax=axes[1])
axes[1].set_title('Purchase Amount by Gender')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Purchase (USD)')

plt.suptitle('')
plt.tight_layout()
plt.savefig('eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Most popular product categories
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(['Product_Category_1','Product_Category_2','Product_Category_3']):
    counts = df[col][df[col] != 0].value_counts().head(10)
    counts.plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Top 10 - {col}')
    axes[i].set_xlabel('Category')
    axes[i].set_ylabel('Count')
plt.tight_layout()
plt.savefig('eda_categories.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Average purchase per category
avg_purchase = df.groupby('Product_Category_1')['Purchase'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
avg_purchase.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Average Purchase Amount per Product Category 1')
ax.set_xlabel('Product Category 1')
ax.set_ylabel('Average Purchase (USD)')
plt.tight_layout()
plt.savefig('eda_avg_purchase.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['Age_Encoded','Gender_Encoded','Occupation','City_Encoded',
                'Stay_Encoded','Marital_Status','Product_Category_1',
                'Product_Category_2','Product_Category_3','Purchase']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('eda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: Purchase vs Occupation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(df['Occupation'], df['Purchase'], alpha=0.3, color='teal')
axes[0].set_title('Purchase vs Occupation')
axes[0].set_xlabel('Occupation Code')
axes[0].set_ylabel('Purchase (USD)')

axes[1].scatter(df['Stay_Encoded'], df['Purchase'], alpha=0.3, color='purple')
axes[1].set_title('Purchase vs Years in City')
axes[1].set_xlabel('Stay in Current City (years)')
axes[1].set_ylabel('Purchase (USD)')

plt.tight_layout()
plt.savefig('eda_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## Stage 4: Clustering Analysis

In [ ]:
# Aggregate per user for clustering
user_df = df.groupby('User_ID').agg(
    Age_Encoded=('Age_Encoded', 'first'),
    Gender_Encoded=('Gender_Encoded', 'first'),
    Occupation=('Occupation', 'first'),
    Marital_Status=('Marital_Status', 'first'),
    Total_Purchase=('Purchase', 'sum'),
    Avg_Purchase=('Purchase', 'mean'),
    Num_Transactions=('Purchase', 'count')
).reset_index()

features = ['Age_Encoded','Gender_Encoded','Occupation','Marital_Status','Avg_Purchase','Num_Transactions']
X = StandardScaler().fit_transform(user_df[features])

# Elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'bo-')
ax.set_title('Elbow Method – Optimal Number of Clusters')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
plt.tight_layout()
plt.savefig('elbow.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fit final model with k=4
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
user_df['Cluster'] = kmeans.fit_predict(X)

# Label clusters based on average spend
cluster_means = user_df.groupby('Cluster')['Avg_Purchase'].mean().sort_values()
labels = {cluster_means.index[0]: 'Budget Shoppers',
          cluster_means.index[1]: 'Casual Buyers',
          cluster_means.index[2]: 'Regular Spenders',
          cluster_means.index[3]: 'Premium Buyers'}
user_df['Cluster_Label'] = user_df['Cluster'].map(labels)
print(user_df.groupby('Cluster_Label')[['Avg_Purchase','Num_Transactions']].mean())

In [ ]:
# Visualize clusters with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
user_df['PCA1'] = X_pca[:, 0]
user_df['PCA2'] = X_pca[:, 1]

colors = {'Budget Shoppers': 'blue', 'Casual Buyers': 'green',
          'Regular Spenders': 'orange', 'Premium Buyers': 'red'}

fig, ax = plt.subplots(figsize=(10, 7))
for label, grp in user_df.groupby('Cluster_Label'):
    ax.scatter(grp['PCA1'], grp['PCA2'], label=label,
               alpha=0.5, s=20, color=colors[label])
ax.set_title('Customer Clusters (PCA)')
ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.legend()
plt.tight_layout()
plt.savefig('clusters.png', dpi=150, bbox_inches='tight')
plt.show()

# Save for Streamlit
user_df.to_csv('user_clusters.csv', index=False)

## Stage 5: Association Rule Mining

In [ ]:
# Build transactions: each user's set of product categories
basket = df.groupby('User_ID').apply(
    lambda x: list(set(
        ['Cat1_' + str(int(c)) for c in x['Product_Category_1'] if c != 0] +
        ['Cat2_' + str(int(c)) for c in x['Product_Category_2'] if c != 0] +
        ['Cat3_' + str(int(c)) for c in x['Product_Category_3'] if c != 0]
    ))
).tolist()

te = TransactionEncoder()
te_array = te.fit_transform(basket)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

# Apriori
frequent_itemsets = apriori(basket_df, min_support=0.1, use_colnames=True)
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.2)
rules = rules.sort_values('lift', ascending=False)
print(f'Generated {len(rules)} association rules')
print(rules[['antecedents','consequents','support','confidence','lift']].head(10))

In [ ]:
# Visualize top rules
top_rules = rules.head(15).copy()
top_rules['rule'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x))) + \
                    ' → ' + top_rules['consequents'].apply(lambda x: ', '.join(list(x)))

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_rules['rule'], top_rules['lift'], color='darkorange')
ax.set_title('Top 15 Association Rules by Lift')
ax.set_xlabel('Lift')
ax.axvline(x=1, color='red', linestyle='--', label='Lift = 1 (random)')
ax.legend()
plt.tight_layout()
plt.savefig('association_rules.png', dpi=150, bbox_inches='tight')
plt.show()

# Save for Streamlit
rules.to_csv('association_rules.csv', index=False)

## Stage 6: Anomaly Detection

In [ ]:
# Z-score method on purchase column
df['Z_Score'] = np.abs(stats.zscore(df['Purchase']))
anomalies = df[df['Z_Score'] > 3].copy()
print(f'Anomalies detected (|z| > 3): {len(anomalies)}')

# IQR method
Q1 = df['Purchase'].quantile(0.25)
Q3 = df['Purchase'].quantile(0.75)
IQR = Q3 - Q1
iqr_anomalies = df[(df['Purchase'] < Q1 - 1.5*IQR) | (df['Purchase'] > Q3 + 1.5*IQR)]
print(f'Anomalies detected (IQR): {len(iqr_anomalies)}')

# Mark anomalies in main df
df['Is_Anomaly'] = df['Z_Score'] > 3
anomaly_summary = anomalies.groupby('Age')['Purchase'].agg(['mean','count']).reset_index()
print('\nAnomaly breakdown by Age:')
print(anomaly_summary)

In [ ]:
# Visualize anomalies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

normal = df[~df['Is_Anomaly']]
anomal = df[df['Is_Anomaly']]

axes[0].scatter(normal.index[:5000], normal['Purchase'][:5000], alpha=0.3, s=5, label='Normal', color='steelblue')
axes[0].scatter(anomal.index, anomal['Purchase'], alpha=0.8, s=20, label='Anomaly', color='red')
axes[0].set_title('Purchase Anomalies (Z-Score > 3)')
axes[0].set_xlabel('Transaction Index')
axes[0].set_ylabel('Purchase (USD)')
axes[0].legend()

axes[1].hist(df['Purchase'], bins=50, color='steelblue', alpha=0.7, label='All')
axes[1].axvline(Q3 + 1.5*IQR, color='red', linestyle='--', label='IQR Upper Bound')
axes[1].set_title('Purchase Distribution with IQR Boundary')
axes[1].set_xlabel('Purchase (USD)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('anomalies.png', dpi=150, bbox_inches='tight')
plt.show()

# Save for Streamlit
df.to_csv('cleaned_data.csv', index=False)
anomalies.to_csv('anomalies.csv', index=False)

## Stage 7: Insights & Reporting

In [ ]:
print('=== KEY INSIGHTS ===')
print()

# 1. Age group spending
age_spend = df.groupby('Age')['Purchase'].mean().sort_values(ascending=False)
print('Top spending age group:', age_spend.index[0], f'(avg ${age_spend.iloc[0]:,.0f})')

# 2. Gender spending
gender_spend = df.groupby('Gender')['Purchase'].mean()
print('Male avg purchase:  $', round(gender_spend.get('M', 0), 2))
print('Female avg purchase: $', round(gender_spend.get('F', 0), 2))

# 3. Most popular category
top_cat = df['Product_Category_1'].value_counts().idxmax()
print(f'Most purchased category: {top_cat}')

# 4. Anomalies
print(f'Unusual high-spenders detected: {len(anomalies)}')
if len(anomalies) > 0:
    top_anomaly = anomalies['Purchase'].max()
    print(f'Highest anomalous purchase: ${top_anomaly:,.0f}')

# 5. Cluster summary
print()
print('Customer segment breakdown:')
print(user_df['Cluster_Label'].value_counts())

## Stage 8: Deployment
See `app.py` — deploy on [Streamlit Cloud](https://streamlit.io/cloud) by:
1. Pushing this repo to GitHub
2. Connecting repo on streamlit.io/cloud
3. Setting main file as `app.py`